In [1]:
# %pip install -q langchain-ollama

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="deepseek-r1:1.5b")  
llm.invoke("What is the capital of France?") # 프롬프트 : LLM호출 

AIMessage(content="The capital of France is Paris. It is the physical location that represents the country's sovereignty and is also associated with the Eiffel Tower, which is named after the city. While there are other significant cities in France, Paris remains the capital.", additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:50:25.0060804Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4844705700, 'load_duration': 79055400, 'prompt_eval_count': 10, 'prompt_eval_duration': 182712700, 'eval_count': 497, 'eval_duration': 4058002500, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e3-2fff-71e0-9882-a7f2a9c7d835-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 497, 'total_tokens': 507})

In [3]:
# ValueError: Invalid input type <class 'int'>. Must be a PromptValue, str, or list of BaseMessages.
# -> invoke 할때 나름의 입력형식이 정해져 있다. 

In [4]:
# PromptValue -> PRomptTemplate을 활용하면 만들 수 있다

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template = "What is the capital of {country}?",  # {country} : placeholder -> 대입을 해줘야 한다. HOW?
    input_variables=["country"],
)

## LLM에 invoke 한 것처럼, 프롬프트도 invoke를 해줘야 한다 
prompt = prompt_template.invoke({"country":"France"})  # prompt 가 promptValue가 됨
print(prompt)  # output : text='What is the capital of France?'

text='What is the capital of France?'


In [6]:
llm.invoke(prompt)  # 잘 들어간다 -> 질문은 France로 대입 된 것
# 즉, invoke가 2번 들어가게 되는 것 (프롬프트 1, LLM 1)

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:50:25.2579857Z', 'done': True, 'done_reason': 'stop', 'total_duration': 180217900, 'load_duration': 80992400, 'prompt_eval_count': 10, 'prompt_eval_duration': 8060400, 'eval_count': 12, 'eval_duration': 62667700, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e3-4334-73f3-9243-37a4130dd5a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22})

#### BaseMessageList

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# llm.invoke([HumanMessage(content="What is the capital of {country}?")]) # 리스트 형식으로 넣을 것

# HumanMessage를 가장 마지막에 넣어서 invoke -> 답변 생성됨
# Few-Shot : 예제를 제공; 답변의 형식에 대한 예제. -> 마치 대화 이력이 있는 것처럼 LLM을 속임 -> 답변을 원하는대로 유도하는 기술
# Few-Shoe 논문 참고
# 복잡한 어플리케이션을 구현할 때는 예제를 주는게 정확도가 좋다
message_list = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of France is Paris."),
    HumanMessage(content="What is the capital of Germany?"),
    AIMessage(content="The capital of France is Berlin."),
    HumanMessage(content="What is the capital of Italy?"),
    AIMessage(content="The capital of France is Rome."),
    HumanMessage(content="What is the capital of {country}?"),
]


llm.invoke(message_list)


AIMessage(content="Hi there! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. For comprehensive details about our models and products, we invite you to consult our official documentation or explore our resources.", additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:50:25.8619986Z', 'done': True, 'done_reason': 'stop', 'total_duration': 593576000, 'load_duration': 109681600, 'prompt_eval_count': 67, 'prompt_eval_duration': 17456800, 'eval_count': 46, 'eval_duration': 416852000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e3-43f3-76b1-98d9-402764be46d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 67, 'output_tokens': 46, 'total_tokens': 113})

- 위와 같은 방법은 Langchain 스럽지 않은 방법 -> ChatPromptTemplate을 활용하자
- ChatPromptTemplate : 채팅 메시지처럼 사용하는 방법이 있음

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# ### 잘못된 방법 -> 아래 셀을 참고하자
# chat_prompt_template = ChatPromptTemplate.from_messages(message_list)  # 이 부분을 수정!
# chat_prompt = chat_prompt_template.invoke({"country" : "France"})   
# print(chat_prompt.messages)   # HumanMessage(content='What is the capital of {country}? -> placeholder에 안들어가있다

In [9]:

chat_prompt_template = ChatPromptTemplate.from_messages([
    # 튜플 형식으로 넣어줘야 한다 - few shot 도 이런 형식으로 활용할 것 (위 방식보다는 LCEL을 구축하는데에 있어 확장성에서도 유리)
    ("system", "You are a helpful assistant"),
    ("human", "What is the capital of {country}?"),
])
chat_prompt = chat_prompt_template.invoke({"country" : "France"})   
print(chat_prompt.messages)   # HumanMessage(content='What is the capital of France? -> 잘 대입됨

[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})]


In [10]:
llm.invoke(chat_prompt)  # 위 셀처럼 해야 LLM 답변도 잘 나온다

AIMessage(content='The capital of France is Paris. Despite the Eiffel Tower serving as the symbol of the capital, Paris itself is the capital city. This makes it the primary location for government buildings and the main city for political purposes.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-03-13T01:50:30.0698449Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4169823800, 'load_duration': 94968800, 'prompt_eval_count': 15, 'prompt_eval_duration': 6711700, 'eval_count': 436, 'eval_duration': 3533259800, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019ce4e3-466a-7220-9ade-f502e60bd177-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 436, 'total_tokens': 451})